In [1]:
# Fine Tuning: Vision Transformers

!pip install -Uq transformers datasets timm accelerate evaluate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.1/12.1 MB 43.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 559.1/559.1 kB 22.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 5.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.1/50.1 MB 18.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 59.8 MB/s eta 0:00:00


In [4]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torchvision.transforms as T

from pathlib import Path
from PIL import Image

import datasets

from transformers.optimization import get_cosine_schedule_with_warmup

from timm import list_models, create_model

from accelerate import Accelerator, notebook_launcher

import evaluate

In [7]:
from torchvision.datasets import VOCDetection

dataset = VOCDetection(
    root="./data",
    year="2007",
    image_set="trainval",
    download=True
)

print("Dataset size:", len(dataset))

100%|██████████| 460M/460M [00:17<00:00, 27.0MB/s]


Dataset size: 5011


In [8]:
class_names = [
    "Aeroplane","Bicycle","Bird","Boat","Bottle",
    "Bus","Car","Cat","Chair","Cow","Diningtable",
    "Dog","Horse","Motorbike","Person",
    "Potted plant","Sheep","Sofa","Train","Tv/monitor"
]

In [9]:
label2id={c:idx for idx,c in enumerate(class_names)}
id2label={idx:c for idx,c in enumerate(class_names)}

In [12]:
import matplotlib.pyplot as plt

def show_samples(dataset, rows=5, cols=5):
    fig, axes = plt.subplots(rows, cols, figsize=(15, 15))
    axes = axes.flatten()

    for i in range(rows * cols):
        image, target = dataset[i]

        axes[i].imshow(image)
        axes[i].axis("off")

    plt.tight_layout()
    plt.show()
show_samples(dataset, rows=5, cols=5)


Output hidden; open in https://colab.research.google.com to view.

In [20]:
# one hot encoding, we are converting this problem into binary classification problem for each label

img_size=(224,224)

train_tfms=T.Compose([
    T.Resize(img_size),
    T.RandomHorizontalFlip(),
    T.RandomRotation(30),
    T.CenterCrop(img_size),
    T.ToTensor(),
    T.Normalize(
        mean=(0.5,0.5,0.5),
        std=(0.5,0.5,0.5)
    )
])



valid_transforms = T.Compose([
    T.Resize(img_size),
    T.ToTensor(),
])

In [34]:
import torchvision.transforms as T

train_transforms = T.Compose([
    T.Resize((224, 224)),
    T.ToTensor(),
])

valid_transforms = T.Compose([
    T.Resize((224, 224)),
    T.ToTensor(),
])


In [35]:
from torchvision.datasets import VOCDetection

train_dataset = VOCDetection(
    root="./data",
    year="2007",
    image_set="train",
    download=True,
    transform=train_transforms
)

valid_dataset = VOCDetection(
    root="./data",
    year="2007",
    image_set="val",
    download=True,
    transform=valid_transforms
)

test_dataset = VOCDetection(
    root="./data",
    year="2007",
    image_set="test",
    download=True,
    transform=valid_transforms
)


In [40]:
import torch

VOC_CLASSES = [
    "aeroplane",
    "bicycle",
    "bird",
    "boat",
    "bottle",
    "bus",
    "car",
    "cat",
    "chair",
    "cow",
    "diningtable",
    "dog",
    "horse",
    "motorbike",
    "person",
    "pottedplant",
    "sheep",
    "sofa",
    "train",
    "tvmonitor",
]

class_to_idx = {
    name: i for i, name in enumerate(VOC_CLASSES)
}


def collate_fn(batch):
    pixel_values = []
    labels = []

    for image, target in batch:

        pixel_values.append(image)

        # 20-class multi-label vector
        label = torch.zeros(20, dtype=torch.float32)

        objects = target["annotation"].get("object", [])

        # If there is only one object, VOC gives a dictionary
        if not isinstance(objects, list):
            objects = [objects]

        for obj in objects:
            class_name = obj["name"]

            if class_name in class_to_idx:
                label[class_to_idx[class_name]] = 1.0

        labels.append(label)

    return {
        "pixel_values": torch.stack(pixel_values),
        "labels": torch.stack(labels)
    }


In [41]:
# handy function to calculate the amount of trainable parametes in model

def param_count(model):
    params = [
        (p.numel(), p.requires_grad)
        for p in model.parameters()
    ]

    total = sum(count for count, _ in params)
    trainable = sum(count for count, is_trainable in params if is_trainable)

    frac = (trainable / total) * 100

    return total, trainable, frac


In [42]:
def train(model_name, batch_size=16, epochs=1, lr=2e-4):
    accelerator = Accelerator()

    train_dl = torch.utils.data.DataLoader(
        train_dataset,
        batch_size=batch_size,
        shuffle=True,
        num_workers=2,
        collate_fn=collate_fn
    )

    valid_dl = torch.utils.data.DataLoader(
        valid_dataset,
        batch_size=batch_size * 2,
        shuffle=False,
        num_workers=2,
        collate_fn=collate_fn
    )

    test_dl = torch.utils.data.DataLoader(
        test_dataset,
        batch_size=batch_size * 2,
        shuffle=False,
        num_workers=2,
        collate_fn=collate_fn
    )

    # timm model
    model = create_model(
        model_name,
        pretrained=True,
        num_classes=20
    )

    total, trainable, frac = param_count(model)
    accelerator.print(
        f"{total=:,} | {trainable=:,} | {frac:.2f}%"
    )

    loss_fn = nn.BCEWithLogitsLoss()

    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=lr,
        weight_decay=0.2
    )

    scheduler = get_cosine_schedule_with_warmup(
        optimizer,
        num_warmup_steps=int(0.1 * len(train_dl)),
        num_training_steps=len(train_dl) * epochs
    )

    model, optimizer, scheduler, train_dl, valid_dl, test_dl = accelerator.prepare(
        model,
        optimizer,
        scheduler,
        train_dl,
        valid_dl,
        test_dl
    )

    for epoch in range(1, epochs + 1):

        # =========================
        # Training
        # =========================
        model.train()

        train_metric = evaluate.load("roc_auc", "multilabel")
        running_loss = 0.0

        for batch in train_dl:

            logits = model(batch["pixel_values"])

            loss = loss_fn(
                logits,
                batch["labels"].float()
            )

            accelerator.backward(loss)

            optimizer.step()
            scheduler.step()
            optimizer.zero_grad()

            running_loss += loss.item()

            logits, labels = accelerator.gather_for_metrics(
                (logits, batch["labels"])
            )

            train_metric.add_batch(
                references=labels,
                prediction_scores=logits
            )

        train_loss = running_loss / len(train_dl)
        train_roc_auc = train_metric.compute(
            average="micro"
        )["roc_auc"]

        accelerator.print(
            f"Epoch {epoch}: "
            f"train_loss={train_loss:.3f} | "
            f"train_roc_auc={train_roc_auc:.3f}"
        )

        # =========================
        # Validation
        # =========================
        model.eval()

        running_loss = 0.0
        valid_metric = evaluate.load("roc_auc", "multilabel")

        for batch in valid_dl:

            with torch.no_grad():
                logits = model(batch["pixel_values"])

            loss = loss_fn(
                logits,
                batch["labels"].float()
            )

            running_loss += loss.item()

            logits, labels = accelerator.gather_for_metrics(
                (logits, batch["labels"])
            )

            valid_metric.add_batch(
                references=labels,
                prediction_scores=logits
            )

        valid_loss = running_loss / len(valid_dl)

        valid_roc_auc = valid_metric.compute(
            average="micro"
        )["roc_auc"]

        accelerator.print(
            f"valid_loss={valid_loss:.3f} | "
            f"valid_roc_auc={valid_roc_auc:.3f}"
        )

        # =========================
        # Save model
        # =========================
        accelerator.save_model(
            model,
            f"./{model_name}-pascal"
        )

    # =========================
    # Testing
    # =========================

    model.eval()

    test_metric = evaluate.load("roc_auc", "multilabel")

    for batch in test_dl:

        with torch.no_grad():
            logits = model(batch["pixel_values"])

        logits, labels = accelerator.gather_for_metrics(
            (logits, batch["labels"])
        )

        test_metric.add_batch(
            references=labels,
            prediction_scores=logits
        )

    test_roc_auc = test_metric.compute(
        average="micro"
    )["roc_auc"]

    accelerator.print(
        f"\nTEST AUROC: {test_roc_auc:.3f}"
    )


In [43]:
model_name="swin_s3_base_224"
notebook_launcher(train,(model_name,8,5,5e-5),num_processes=2)

Launching training on one GPU.
total=70,372,142 | trainable=70,372,142 | 100.00%
Epoch 1: train_loss=0.164 | train_roc_auc=0.921
valid_loss=0.071 | valid_roc_auc=0.983
Epoch 2: train_loss=0.057 | train_roc_auc=0.991
valid_loss=0.061 | valid_roc_auc=0.987
Epoch 3: train_loss=0.031 | train_roc_auc=0.998
valid_loss=0.055 | valid_roc_auc=0.990
Epoch 4: train_loss=0.017 | train_roc_auc=1.000
valid_loss=0.053 | valid_roc_auc=0.990
Epoch 5: train_loss=0.013 | train_roc_auc=1.000
valid_loss=0.053 | valid_roc_auc=0.990

TEST AUROC: 0.990


In [45]:
from timm import create_model

num_classes = 20

model = create_model(
    model_name,
    pretrained=True,
    num_classes=num_classes
)

print(model)


SwinTransformer(
  (patch_embed): PatchEmbed(
    (proj): Conv2d(3, 96, kernel_size=(4, 4), stride=(4, 4))
    (norm): LayerNorm((96,), eps=1e-05, elementwise_affine=True)
  )
  (layers): Sequential(
    (0): SwinTransformerStage(
      (downsample): Identity()
      (blocks): Sequential(
        (0): SwinTransformerBlock(
          (norm1): LayerNorm((96,), eps=1e-05, elementwise_affine=True)
          (attn): WindowAttention(
            (qkv): Linear(in_features=96, out_features=288, bias=True)
            (attn_drop): Dropout(p=0.0, inplace=False)
            (proj): Linear(in_features=96, out_features=96, bias=True)
            (proj_drop): Dropout(p=0.0, inplace=False)
            (softmax): Softmax(dim=-1)
          )
          (drop_path1): Identity()
          (norm2): LayerNorm((96,), eps=1e-05, elementwise_affine=True)
          (mlp): Mlp(
            (fc1): Linear(in_features=96, out_features=384, bias=True)
            (act): GELU(approximate='none')
            (drop1): 

In [46]:
from safetensors.torch import load_model

In [49]:
load_model(
    model,
    f"./{model_name}-pascal/model.safetensors"
)

(set(), [])